# Gridiron Intelligence: Interactive Scouting Workbench

This notebook mimics the app workflow with interactive controls:

1. Select **year** (`2026`, `2027`, `2028`)
2. Select **player** (searchable by `name | position | high_school`)
3. Select **target college team**
4. Click **Submit** to run retrieval + enrichment + synthesis

Pipeline on submit:
- SQL retrieval by `recruit_id` from Supabase
- Clean scouting profile (drop blank `skill_*`, drop `flag_* = 0`)
- Preserve full `hs_athletic_background`
- DuckDuckGo search + Gemini `gemini-2.5-flash-lite` summary
- Vector insights lookup (skip gracefully if RPC/dependencies unavailable)
- Visual model score/probability card
- Final synthesis with Gemini `gemini-3-flash`

In [1]:
from __future__ import annotations

import ast
import json
import os
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import HTML, Markdown, display

try:
    from dotenv import load_dotenv
except Exception:
    load_dotenv = None

try:
    from supabase import create_client
except Exception:
    create_client = None

try:
    from langchain_google_genai import ChatGoogleGenerativeAI
except Exception:
    ChatGoogleGenerativeAI = None

try:
    from ddgs import DDGS
except Exception:
    DDGS = None

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 180)

In [2]:
import google.generativeai as genai
import os
from dotenv import load_dotenv

# Load environment variables from the .env file in the workspace root
load_dotenv(dotenv_path="../GEMINI_API_KEY.env")
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    # Try local .env if root one fails
    load_dotenv()
    api_key = os.getenv("GEMINI_API_KEY")

if api_key:
    genai.configure(api_key=api_key)
    print("Gemini API configured successfully.")
else:
    print("Error: GEMINI_API_KEY not found in environment.")

# List all available models
print("\nAvailable Gemini Models:")
try:
    for m in genai.list_models():
        if 'generateContent' in m.supported_generation_methods:
            print(f"Gen Model: {m.name}")
        if 'embedContent' in m.supported_generation_methods:
            print(f"Embed Model: {m.name}")
except Exception as e:
    print(f"Error listing models: {e}")

C:\Users\ekerv\AppData\Local\Temp\ipykernel_13112\620047395.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Gemini API configured successfully.

Available Gemini Models:
Gen Model: models/gemini-2.5-flash
Gen Model: models/gemini-2.5-pro
Gen Model: models/gemini-2.0-flash
Gen Model: models/gemini-2.0-flash-001
Gen Model: models/gemini-2.0-flash-exp-image-generation
Gen Model: models/gemini-2.0-flash-lite-001
Gen Model: models/gemini-2.0-flash-lite
Gen Model: models/gemini-2.5-flash-preview-tts
Gen Model: models/gemini-2.5-pro-preview-tts
Gen Model: models/gemma-3-1b-it
Gen Model: models/gemma-3-4b-it
Gen Model: models/gemma-3-12b-it
Gen Model: models/gemma-3-27b-it
Gen Model: models/gemma-3n-e4b-it
Gen Model: models/gemma-3n-e2b-it
Gen Model: models/gemini-flash-latest
Gen Model: models/gemini-flash-lite-latest
Gen Model: models/gemini-pro-latest
Gen Model: models/gemini-2.5-flash-lite
Gen Model: models/gemini-2.5-flash-image
Gen Model: models/gemini-2.5-flash-lite-preview-09-2025
Gen Model: models/gemini-3-pro-preview
Gen Model: models/gemini-3-flash-preview
Gen Model: models/gemini-3.1-pro

In [3]:
def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "data" / "modeling_datasets").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing data/modeling_datasets")

PROJECT_ROOT = resolve_project_root()
RECRUITS_PATH = PROJECT_ROOT / "data" / "modeling_datasets" / "recruits" / "master_recruits_2015_2028.csv"
MODEL_TIERS_PATH = PROJECT_ROOT / "data" / "modeling_datasets" / "final" / "models" / "input_data" / "Model_Tiers.csv"

if load_dotenv is not None:
    supabase_env = PROJECT_ROOT / "SUPABASE.env"
    gemini_env = PROJECT_ROOT / "GEMINI_API_KEY.env"
    if supabase_env.exists():
        load_dotenv(supabase_env, override=False)
    if gemini_env.exists():
        load_dotenv(gemini_env, override=False)

CONFIG = {
    "SUPABASE_URL": os.getenv("SUPABASE_URL", ""),
    "SUPABASE_SERVICE_ROLE_KEY": os.getenv("SUPABASE_SERVICE_ROLE_KEY", ""),
    "GEMINI_API_KEY": os.getenv("GEMINI_API_KEY", ""),
    "YEARS": [2026, 2027, 2028],
    "FINAL_MODEL": "gemini-3-flash-preview",
    "SUMMARY_MODEL": "gemini-2.5-flash-lite",
    "VECTOR_MATCH_COUNT": 6,
    "VECTOR_MATCH_THRESHOLD": 0.15,
    "VECTOR_RPC_NAME": "match_gi_factoids",
}

TABLES = {
    "player_master": "gi_player_master",
    "scouting_features": "gi_scouting_report_features",
    "pred_score": "gi_model_prediction_score",
    "pred_threshold": "gi_model_prediction_thresholds",
    "factoid_vectors": "gi_factoid_vectors",
}

TARGET_TEAMS = [
    "Alabama", "Auburn", "Clemson", "Colorado", "Duke", "Florida", "Florida State",
    "Georgia", "Georgia Tech", "LSU", "Miami", "Michigan", "NC State", "Notre Dame",
    "Ohio State", "Ole Miss", "Oregon", "South Carolina", "Tennessee", "Texas",
    "Texas A&M", "Charlotte", "USC", "Virginia Tech", "Wake Forest"
]

TARGET_SEARCH_SITES = ["maxpreps.com", "247sports.com", "rivals.com", "espn.com", "on3.com"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Recruit file exists: {RECRUITS_PATH.exists()}")
print(f"Model tiers file exists: {MODEL_TIERS_PATH.exists()}")
print(f"Supabase configured: {CONFIG['SUPABASE_URL'].startswith('http') and bool(CONFIG['SUPABASE_SERVICE_ROLE_KEY'])}")
print(f"Gemini key configured: {bool(CONFIG['GEMINI_API_KEY'])}")
print(f"Vector RPC: {CONFIG['VECTOR_RPC_NAME']}")

Project root: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence
Recruit file exists: True
Model tiers file exists: True
Supabase configured: True
Gemini key configured: True
Vector RPC: match_gi_factoids


In [4]:
def get_supabase_client():
    if create_client is None:
        return None
    if not CONFIG['SUPABASE_URL'] or not CONFIG['SUPABASE_SERVICE_ROLE_KEY']:
        return None
    return create_client(CONFIG['SUPABASE_URL'], CONFIG['SUPABASE_SERVICE_ROLE_KEY'])

def get_llm(model_name: str, temperature: float = 0.2, max_output_tokens: int = 1800):
    if ChatGoogleGenerativeAI is None or not CONFIG['GEMINI_API_KEY']:
        return None
    return ChatGoogleGenerativeAI(
        model=model_name,
        google_api_key=CONFIG['GEMINI_API_KEY'],
        temperature=temperature,
        max_output_tokens=max_output_tokens,
    )

EMBED_MODEL = None
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
EMBED_MODEL_LOAD_ERROR = None

POS_MAP = {
    "CB": "DB", "S": "DB", "FS": "DB", "SS": "DB", "DB": "DB",
    "DE": "EDGE", "EDGE": "EDGE",
    "DT": "IDL", "NT": "IDL", "DL": "IDL",
    "LB": "LB", "OLB": "LB", "ILB": "LB", "MLB": "LB",
    "OL": "OL", "OT": "OL", "OG": "OL", "C": "OL",
    "QB": "QB", "PRO": "QB", "DUAL": "QB",
    "RB": "RB", "HB": "RB", "FB": "RB",
    "K": "SPEC", "P": "SPEC", "PK": "SPEC", "LS": "SPEC", "RET": "SPEC", "SPEC": "SPEC",
    "TE": "TE",
    "WR": "WR",
}

def normalize_position_group(position_value: str | None) -> str:
    raw = str(position_value or '').strip().upper()
    if not raw:
        return ''
    return POS_MAP.get(raw, raw)

def get_embedding_model():
    global EMBED_MODEL, EMBED_MODEL_LOAD_ERROR

    if EMBED_MODEL is not None:
        return EMBED_MODEL

    if EMBED_MODEL_LOAD_ERROR is not None:
        raise RuntimeError(EMBED_MODEL_LOAD_ERROR)

    try:
        from sentence_transformers import SentenceTransformer
        EMBED_MODEL = SentenceTransformer(EMBED_MODEL_NAME)
        return EMBED_MODEL
    except Exception as exc:
        EMBED_MODEL_LOAD_ERROR = f"Embedding model load failed: {exc}"
        raise RuntimeError(EMBED_MODEL_LOAD_ERROR)

def llm_response_to_text(response: Any) -> str:
    if response is None:
        return ""

    if isinstance(response, str):
        return response

    content = getattr(response, 'content', response)

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                txt = item.get('text') or item.get('output_text') or ''
                if txt:
                    parts.append(str(txt))
            else:
                txt = getattr(item, 'text', None)
                if txt:
                    parts.append(str(txt))
        if parts:
            return "\n".join(parts)
        return str(content)

    if isinstance(content, dict):
        txt = content.get('text') or content.get('output_text')
        if txt:
            return str(txt)

    return str(content)

def to_float_or_none(value: Any) -> float | None:
    try:
        if value is None:
            return None
        if pd.isna(value):
            return None
        return float(value)
    except Exception:
        return None

def parse_jsonish(value: Any) -> dict:
    if isinstance(value, dict):
        return value
    if value is None:
        return {}
    text = str(value).strip()
    if not text or text.lower() in {"none", "nan"}:
        return {}
    try:
        parsed = json.loads(text)
        return parsed if isinstance(parsed, dict) else {}
    except Exception:
        try:
            parsed = ast.literal_eval(text)
            return parsed if isinstance(parsed, dict) else {}
        except Exception:
            return {}

def first_non_null(row: dict, candidates: list[str]):
    for key in candidates:
        value = row.get(key)
        if value is None:
            continue
        if isinstance(value, str) and value.strip() == "":
            continue
        if isinstance(value, float) and pd.isna(value):
            continue
        return value
    return None

def _parse_score_range(range_text: str) -> tuple[float, float]:
    import re
    text = str(range_text).replace("–", "-").replace("—", "-").strip()
    nums = re.findall(r"\d+(?:\.\d+)?", text)
    if len(nums) < 2:
        return (np.nan, np.nan)
    return float(nums[0]), float(nums[1])

def load_model_tiers() -> pd.DataFrame:
    if not MODEL_TIERS_PATH.exists():
        return pd.DataFrame(columns=["Score Range", "Career Designation", "College Outlook", "Professional Outlook", "low", "high"] )
    df = pd.read_csv(MODEL_TIERS_PATH)
    for c in ["Score Range", "Career Designation", "College Outlook", "Professional Outlook"]:
        if c not in df.columns:
            df[c] = ""
    bounds = df["Score Range"].apply(_parse_score_range)
    df["low"] = bounds.apply(lambda x: x[0])
    df["high"] = bounds.apply(lambda x: x[1])
    return df.sort_values(["low", "high"]).reset_index(drop=True)

MODEL_TIERS_DF = load_model_tiers()

def score_tier(score: float | None) -> str:
    if score is None or MODEL_TIERS_DF.empty:
        return 'Unknown'
    for _, row in MODEL_TIERS_DF.iterrows():
        low = to_float_or_none(row.get('low'))
        high = to_float_or_none(row.get('high'))
        if low is None or high is None:
            continue
        if low <= score <= high:
            return str(row.get('Career Designation', 'Unknown'))
    return 'Unknown'

def tier_definitions_markdown() -> str:
    if MODEL_TIERS_DF.empty:
        return "Tier definitions unavailable."
    lines = []
    for _, row in MODEL_TIERS_DF.iterrows():
        lines.append(
            f"- **{row.get('Career Designation', '')}** ({row.get('Score Range', '')}): "
            f"College Outlook: {row.get('College Outlook', '')}; "
            f"Professional Outlook: {row.get('Professional Outlook', '')}"
        )
    return "\n".join(lines)

def merge_scouting_sources(scouting_row: dict) -> dict:
    merged = parse_jsonish(scouting_row.get('scouting_json'))
    if not isinstance(merged, dict):
        merged = {}
    for key, value in scouting_row.items():
        if str(key).startswith('skill_') or str(key).startswith('flag_'):
            if key not in merged or merged.get(key) in (None, '', 'none', 'nan'):
                merged[key] = value
    return merged

def clean_scouting_profile(scouting_json: dict) -> dict:
    cleaned = {}
    for key, value in scouting_json.items():
        if key.startswith('skill_'):
            if value is None:
                continue
            text_value = str(value).strip()
            if text_value == '' or text_value.lower() in {'none', 'nan'}:
                continue
            cleaned[key] = value
        elif key.startswith('flag_'):
            num = to_float_or_none(value)
            if num is None or num == 0:
                continue
            cleaned[key] = int(num) if float(num).is_integer() else num
        else:
            if value is None:
                continue
            if isinstance(value, float) and pd.isna(value):
                continue
            cleaned[key] = value
    return cleaned

def build_player_profile_view(player_row: dict) -> dict:
    return {
        'recruit_id': first_non_null(player_row, ['recruit_id', 'player_id']),
        'player_name': first_non_null(player_row, ['player_name', 'name']),
        'position': first_non_null(player_row, ['position']),
        'rating': first_non_null(player_row, ['rating']),
        'height_raw': first_non_null(player_row, ['height_raw', 'height']),
        'weight_raw': first_non_null(player_row, ['weight_raw', 'weight']),
        'height_inches': first_non_null(player_row, ['height_inches']),
        'weight_lbs': first_non_null(player_row, ['weight_lbs']),
        'high_school': first_non_null(player_row, ['high_school']),
        'city': first_non_null(player_row, ['city']),
        'state': first_non_null(player_row, ['state']),
        'committed_to': first_non_null(player_row, ['committed_to']),
        'year': first_non_null(player_row, ['year']),
    }

In [5]:
def load_player_index() -> pd.DataFrame:
    if not RECRUITS_PATH.exists():
        raise FileNotFoundError(f'Missing recruit source file: {RECRUITS_PATH}')

    df = pd.read_csv(RECRUITS_PATH)
    if 'recruit_id' not in df.columns and 'player_id' in df.columns:
        df['recruit_id'] = df['player_id']

    required = ['recruit_id', 'name', 'position', 'high_school', 'year']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f'Recruit file missing columns: {missing}')

    df = df[required + [c for c in ['committed_to'] if c in df.columns]].copy()
    df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
    df = df[df['year'].isin(CONFIG['YEARS'])].copy()

    for col in ['recruit_id', 'name', 'position', 'high_school']:
        df[col] = df[col].astype(str).str.strip()

    df = df[df['recruit_id'] != ''].drop_duplicates(subset=['recruit_id'])
    df['player_label'] = df.apply(lambda r: f"{r['name']} | {r['position']} | {r['high_school']}", axis=1)
    return df.sort_values(['year', 'name']).reset_index(drop=True)

PLAYER_INDEX = load_player_index()
print(f'Player index rows (2026-2028): {len(PLAYER_INDEX):,}')
display(PLAYER_INDEX.head(5))

Player index rows (2026-2028): 4,149


,recruit_id,name,position,high_school,year,committed_to,player_label
0,202601982,A'Davion Dale,CB,Martha Layne Collins,2026,Uncommitted,A'Davion Dale | CB | Martha Layne Collins
1,202601874,AJ Logan,WR,Mount Miguel,2026,Boise State,AJ Logan | WR | Mount Miguel
2,202601031,AJ Marks,CB,IMG Academy,2026,Wake Forest,AJ Marks | CB | IMG Academy
3,202602900,AJ Morton,CB,Bellevue,2026,Uncommitted,AJ Morton | CB | Bellevue
4,202602727,AJ Rangel,WR,Santa Teresa,2026,Uncommitted,AJ Rangel | WR | Santa Teresa


In [7]:
def fetch_player_bundle(sb, recruit_id: str) -> dict:
    if sb is None:
        raise RuntimeError('Supabase client is not configured. Check SUPABASE.env values.')

    recruit_id = str(recruit_id).strip()

    player_master = sb.table(TABLES['player_master']).select('*').eq('recruit_id', recruit_id).limit(1).execute().data
    scouting = sb.table(TABLES['scouting_features']).select('*').eq('recruit_id', recruit_id).limit(1).execute().data

    pred_score = (
        sb.table(TABLES['pred_score'])
        .select('*')
        .eq('recruit_id', recruit_id)
        .order('as_of_date', desc=True)
        .order('updated_at', desc=True)
        .limit(1)
        .execute()
        .data
    )

    pred_threshold = (
        sb.table(TABLES['pred_threshold'])
        .select('*')
        .eq('recruit_id', recruit_id)
        .order('as_of_date', desc=True)
        .order('updated_at', desc=True)
        .limit(1)
        .execute()
        .data
    )

    player_row = player_master[0] if player_master else {}
    scouting_row = scouting[0] if scouting else {}
    pred_score_row = pred_score[0] if pred_score else {}
    pred_thr_row = pred_threshold[0] if pred_threshold else {}

    scouting_merged = merge_scouting_sources(scouting_row)
    scouting_clean = clean_scouting_profile(scouting_merged)
    player_profile = build_player_profile_view(player_row)

    return {
        'recruit_id': recruit_id,
        'player': player_row,
        'player_profile': player_profile,
        'scouting_raw': scouting_row,
        'scouting_merged': scouting_merged,
        'scouting_clean': scouting_clean,
        'pred_score': pred_score_row,
        'pred_threshold': pred_thr_row,
    }

def duckduckgo_search(player_name: str, position: str, high_school: str, year: int, max_results: int = 12) -> list[dict]:
    if DDGS is None:
        return []

    query = (
        f"{player_name} {position} {high_school} {year} football recruiting "
        f"(site:maxpreps.com OR site:247sports.com OR site:rivals.com OR site:espn.com OR site:on3.com)"
    )

    rows = []
    with DDGS() as ddgs:
        for result in ddgs.text(query, max_results=max_results):
            url = result.get('href', '') or ''
            if not any(site in url for site in TARGET_SEARCH_SITES):
                continue
            rows.append({
                'title': result.get('title', ''),
                'url': url,
                'snippet': result.get('body', ''),
            })
    return rows

def summarize_web_with_flash_lite(player_name: str, position: str, search_rows: list[dict]) -> str:
    if not search_rows:
        return 'No relevant web articles were found from target recruiting sites.'

    llm = get_llm(CONFIG['SUMMARY_MODEL'], temperature=0.0, max_output_tokens=1200)
    if llm is None:
        return 'Gemini summary skipped: API key/model client not configured.'

    context_chunks = []
    for i, row in enumerate(search_rows[:10], start=1):
        context_chunks.append(
            f"[{i}] Title: {row['title']}\nURL: {row['url']}\nSnippet: {row['snippet']}"
        )

    prompt = f"""
You are a recruiting research assistant. Summarize recent web intelligence for {player_name} ({position}).
Only use the provided sources. Do not invent facts.

Output format:
1) Key facts (4-6 bullets)
2) Recruiting/status updates (2-4 bullets)
3) Source list (title + url)

Sources:
{chr(10).join(context_chunks)}
"""

    response = llm.invoke(prompt)
    return llm_response_to_text(response)

def _call_vector_rpc(sb, embedding: list[float], filter_position: str, filter_state: str | None, threshold: float, top_k: int) -> list[dict]:
    rpc_payload = {
        'query_embedding': embedding,
        'match_threshold': float(threshold),
        'match_count': int(top_k),
        'filter_position': filter_position,
    }
    if filter_state:
        rpc_payload['filter_state'] = filter_state

    try:
        return sb.rpc(CONFIG['VECTOR_RPC_NAME'], rpc_payload).execute().data or []
    except Exception as exc:
        message = str(exc)
        if ('filter_state' in message) and ('Could not find the function' in message):
            rpc_payload.pop('filter_state', None)
            return sb.rpc(CONFIG['VECTOR_RPC_NAME'], rpc_payload).execute().data or []
        raise

def vector_insights_query(
    sb,
    query_text: str,
    position: str | None = None,
    state: str | None = None,
    top_k: int = 6,
    threshold: float | None = None,
) -> dict:
    if sb is None:
        return {'status': 'skipped', 'reason': 'Supabase client unavailable', 'insights': []}

    pos = (position or '').strip().upper()
    st = (state or '').strip().upper()

    if not pos:
        return {'status': 'skipped', 'reason': 'Missing player position for exact positional filter', 'insights': []}

    try:
        model = get_embedding_model()
    except Exception as exc:
        return {'status': 'skipped', 'reason': str(exc), 'insights': []}

    try:
        embedding = model.encode([query_text], normalize_embeddings=True)[0].tolist()
        use_threshold = float(threshold if threshold is not None else CONFIG['VECTOR_MATCH_THRESHOLD'])

        result_rows = _call_vector_rpc(
            sb=sb,
            embedding=embedding,
            filter_position=pos,
            filter_state=st or None,
            threshold=use_threshold,
            top_k=top_k,
        )

        used_mode = f"exact position={pos}{f', state={st}' if st else ''}, threshold={use_threshold:.3f}"

        insights = []
        for row in result_rows:
            text = row.get('factoid_text') or ''
            factoid_type = row.get('factoid_type') or 'insight'
            similarity = to_float_or_none(row.get('similarity'))
            if not text:
                continue
            if similarity is None:
                insights.append(f"[{factoid_type}] {text}")
            else:
                insights.append(f"[{factoid_type} | sim={similarity:.3f}] {text}")

        if not insights:
            return {
                'status': 'ok',
                'reason': f'No matches found ({used_mode})',
                'insights': [],
            }

        return {
            'status': 'ok',
            'reason': used_mode,
            'insights': insights[:top_k],
        }
    except Exception as exc:
        return {'status': 'skipped', 'reason': f"Vector RPC unavailable: {exc}", 'insights': []}

In [8]:
def build_score_card_html(pred_score: dict, pred_threshold: dict) -> str:
    score = to_float_or_none(pred_score.get('predictive_score_0_100'))
    prob_ge20 = to_float_or_none(pred_threshold.get('prob_ge20'))
    prob_ge50 = to_float_or_none(pred_threshold.get('prob_ge50'))
    prob_ge80 = to_float_or_none(pred_threshold.get('prob_ge80'))

    score_display = 'N/A' if score is None else f'{score:.1f}'
    tier = score_tier(score)

    def pct(x: float | None) -> str:
        return 'N/A' if x is None else f'{(x * 100):.1f}%'

    def bar(x: float | None) -> str:
        if x is None:
            width = 0
        else:
            width = max(0, min(100, x * 100))
        return (
            f"<div style='background:#e9ecef;border-radius:8px;height:14px;'>"
            f"<div style='width:{width:.1f}%;background:#0d6efd;height:14px;border-radius:8px;'></div>"
            f"</div>"
        )

    score_width = 0 if score is None else max(0, min(100, score))

    html = f"""
<div style="border:1px solid #d9d9d9;border-radius:12px;padding:14px 16px;margin:8px 0 16px 0;background:#ffffff;">
  <h3 style="margin:0 0 8px 0;">Model Output</h3>
  <div style="display:flex;gap:20px;flex-wrap:wrap;">
    <div style="min-width:240px;">
      <div style="font-size:13px;color:#666;">Predictive Score</div>
      <div style="font-size:28px;font-weight:700;line-height:1.1;">{score_display}/100</div>
      <div style="font-size:13px;color:#444;margin-bottom:8px;">Career Designation: <b>{tier}</b></div>
      <div style='background:#e9ecef;border-radius:8px;height:16px;'>
        <div style='width:{score_width:.1f}%;background:#198754;height:16px;border-radius:8px;'></div>
      </div>
      <div style="display:flex;justify-content:space-between;font-size:11px;color:#666;margin-top:4px;">
        <span>0</span><span>20</span><span>40</span><span>60</span><span>80</span><span>100</span>
      </div>
    </div>
    <div style="min-width:320px;flex:1;">
      <div style="margin:0 0 8px 0;font-size:13px;color:#666;">Classification Threshold Probabilities</div>
      <div style="margin-bottom:10px;"><b>Contributor (>20):</b> {pct(prob_ge20)} {bar(prob_ge20)}</div>
      <div style="margin-bottom:10px;"><b>Multi-Year Starter (>50):</b> {pct(prob_ge50)} {bar(prob_ge50)}</div>
      <div><b>Elite (>80):</b> {pct(prob_ge80)} {bar(prob_ge80)}</div>
    </div>
  </div>
</div>
"""
    return html

def build_final_prompt(
    year: int,
    target_team: str,
    player_row: dict,
    scouting_clean: dict,
    hs_athletic_background: str,
    pred_score_row: dict,
    pred_thr_row: dict,
    web_summary: str,
    vector_result: dict,
) -> str:
    vector_text = '\n'.join([f"- {x}" for x in vector_result.get('insights', [])])
    if not vector_text:
        vector_text = f"No vector insights available ({vector_result.get('reason', 'not returned')})."

    tier_defs = tier_definitions_markdown()

    return f"""
You are a senior college football recruiting scout.

Generate a structured scouting report using only the provided context.

## Request Context
- Recruiting Year: {year}
- Target Team: {target_team}

## Player Profile
{json.dumps(player_row, indent=2, default=str)}

## High School Athletic Background (full text)
{hs_athletic_background}

## Filtered Scouting Attributes
{json.dumps(scouting_clean, indent=2, default=str)}

## Model Outputs
Score Row:
{json.dumps(pred_score_row, indent=2, default=str)}

Threshold Row (classification probabilities only):
{json.dumps(pred_thr_row, indent=2, default=str)}

## Official Tier Definitions (Career Designation + Outlooks)
Use these as the canonical interpretation of score bands:
{tier_defs}

## Web Intelligence Summary (DuckDuckGo + Gemini 2.5-flash-lite)
{web_summary}

## Historical/Vector Insights
{vector_text}

## Output Format
1) Player Snapshot (5-8 bullets)
2) Trait-Based Evaluation (strengths, risks, development focus)
3) Fit for {target_team} (scheme/roster/development rationale)
4) Career Designation Mapping (map model score to the official designation and explain college + pro outlook implications)
5) Threshold Probability Interpretation (P>=20 (Contributor) / P>=50 (Multi-Year Starter) / P>=80 (Elite Prospect) - explain what these probabilities mean in practical terms)
6) Final Recommendation (clear go/no-go + confidence level)

Do not mention missing tools or pipeline internals.
"""

def run_final_synthesis(prompt: str) -> str:
    llm = get_llm(CONFIG['FINAL_MODEL'], temperature=0.25, max_output_tokens=2200)
    if llm is None:
        return 'Final synthesis skipped: Gemini client not configured.'
    response = llm.invoke(prompt)
    return llm_response_to_text(response)

In [9]:
# Optional: preload embedding model once to avoid first-submit delay
try:
    _ = get_embedding_model()
    print(f"Embedding model loaded once: {EMBED_MODEL_NAME}")
except Exception as exc:
    print(f"Embedding preload skipped: {exc}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded once: all-MiniLM-L6-v2


In [10]:
def prepare_player_options(selected_year: int) -> tuple[list[str], dict[str, str], str | None]:
    df = PLAYER_INDEX[PLAYER_INDEX['year'] == selected_year].copy()
    labels = df['player_label'].tolist()
    label_to_id = dict(zip(df['player_label'], df['recruit_id']))
    default_label = labels[0] if labels else None
    return labels, label_to_id, default_label

year_dropdown = widgets.Dropdown(
    options=CONFIG['YEARS'],
    value=CONFIG['YEARS'][0],
    description='Year:',
    style={'description_width': 'initial'},
)

player_combobox = widgets.Combobox(
    placeholder='Type name, position, or high school...',
    description='Player:',
    ensure_option=True,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='760px'),
)

team_dropdown = widgets.Dropdown(
    options=TARGET_TEAMS,
    value=TARGET_TEAMS[0],
    description='Target Team:',
    style={'description_width': 'initial'},
)

submit_btn = widgets.Button(
    description='Submit',
    button_style='primary',
    icon='play'
)

status_html = widgets.HTML(value='')
output = widgets.Output()

PLAYER_LOOKUP: dict[str, str] = {}

def refresh_players(year_value: int):
    global PLAYER_LOOKUP
    labels, mapping, default_label = prepare_player_options(year_value)
    PLAYER_LOOKUP = mapping
    player_combobox.options = labels
    player_combobox.value = default_label or ''

def on_year_change(change):
    if change['name'] == 'value' and change['new'] is not None:
        refresh_players(int(change['new']))

year_dropdown.observe(on_year_change, names='value')
refresh_players(int(year_dropdown.value))

controls = widgets.VBox([
    widgets.HBox([year_dropdown, team_dropdown]),
    player_combobox,
    submit_btn,
    status_html,
])

In [11]:
def on_submit(_):
    output.clear_output()

    selected_year = int(year_dropdown.value)
    selected_label = str(player_combobox.value).strip()
    target_team = str(team_dropdown.value).strip()

    recruit_id = PLAYER_LOOKUP.get(selected_label)
    if not recruit_id:
        status_html.value = "<span style='color:#b91c1c;font-weight:600;'>Pick a valid player from the searchable dropdown list.</span>"
        return

    status_html.value = "<span style='color:#1d4ed8;font-weight:600;'>Running pipeline... retrieving SQL profile, web summary, vector insights, and final synthesis.</span>"

    sb = get_supabase_client()

    try:
        bundle = fetch_player_bundle(sb, recruit_id)
    except Exception as exc:
        status_html.value = f"<span style='color:#b91c1c;font-weight:600;'>Data retrieval failed: {exc}</span>"
        return

    player_row = bundle.get('player', {})
    player_profile = bundle.get('player_profile', {})
    scouting_row = bundle.get('scouting_raw', {})
    scouting_clean = bundle.get('scouting_clean', {})
    pred_score_row = bundle.get('pred_score', {})
    pred_thr_row = bundle.get('pred_threshold', {})

    player_name = player_profile.get('player_name') or selected_label.split('|')[0].strip()
    position = player_profile.get('position') or ''
    vector_position = normalize_position_group(position)
    player_state = str(player_profile.get('state') or '').strip().upper()
    high_school = player_profile.get('high_school') or ''
    hs_athletic_background = scouting_row.get('hs_athletic_background') or ''

    search_rows = duckduckgo_search(player_name, position, high_school, selected_year, max_results=12)
    web_summary = summarize_web_with_flash_lite(player_name, position, search_rows)

    vector_query_text = (
        f"Player: {player_name}\n"
        f"Position: {position}\n"
        f"Position Group: {vector_position}\n"
        f"State: {player_state}\n"
        f"High School: {high_school}\n"
        f"Measurables: height_raw={player_profile.get('height_raw')}, weight_raw={player_profile.get('weight_raw')}, "
        f"height_inches={player_profile.get('height_inches')}, weight_lbs={player_profile.get('weight_lbs')}\n"
        f"Rating: {player_profile.get('rating')}\n\n"
        f"HS Athletic Background:\n{hs_athletic_background}\n\n"
        f"Filtered scouting report:\n{json.dumps(scouting_clean, default=str)}\n\n"
        f"Flash 2.5 summary:\n{web_summary}"
    )

    vector_result = vector_insights_query(
        sb=sb,
        query_text=vector_query_text,
        position=vector_position,
        state=player_state,
        top_k=CONFIG['VECTOR_MATCH_COUNT'],
        threshold=CONFIG['VECTOR_MATCH_THRESHOLD'],
    )

    final_prompt = build_final_prompt(
        year=selected_year,
        target_team=target_team,
        player_row=player_profile if player_profile else player_row,
        scouting_clean=scouting_clean,
        hs_athletic_background=hs_athletic_background,
        pred_score_row=pred_score_row,
        pred_thr_row=pred_thr_row,
        web_summary=web_summary,
        vector_result=vector_result,
    )

    final_report = run_final_synthesis(final_prompt)

    with output:
        display(Markdown(f"## Scouting Workbench Output — {player_name}"))
        display(Markdown(f"- Recruit ID: `{recruit_id}`  \\ - Year: `{selected_year}`  \\ - Target Team: `{target_team}`"))

        display(Markdown("### Player Profile (from gi_player_master)"))
        display(Markdown(f"```json\n{json.dumps(player_profile, indent=2, default=str)}\n```"))

        display(HTML(build_score_card_html(pred_score_row, pred_thr_row)))

        display(Markdown("### Filtered Scouting Profile"))
        display(Markdown(f"```json\n{json.dumps(scouting_clean, indent=2, default=str)}\n```"))

        display(Markdown("### Full HS Athletic Background"))
        display(Markdown(hs_athletic_background if hs_athletic_background else "(No HS athletic background found)"))

        display(Markdown("### Web Summary (Gemini 2.5 Flash Lite)"))
        display(Markdown(web_summary))

        vector_mode = vector_result.get('reason', '')
        vector_header = "### Vector Insights"
        if vector_mode:
            vector_header += f" ({vector_mode})"
        if vector_result.get('status') != 'ok':
            vector_header += f" (Skipped: {vector_result.get('reason', 'unavailable')})"
        display(Markdown(vector_header))
        if vector_result.get('insights'):
            for i, insight in enumerate(vector_result['insights'], start=1):
                display(Markdown(f"{i}. {insight}"))
        else:
            display(Markdown("No vector insights returned."))

        display(Markdown("### Final Scout Synthesis (Gemini 3 Flash)"))
        display(Markdown(final_report))

    status_html.value = "<span style='color:#166534;font-weight:600;'>Completed.</span>"

submit_btn.on_click(on_submit)
print('Interactive callback registered. Select inputs above and click Submit.')

Interactive callback registered. Select inputs above and click Submit.


In [12]:
display(controls)
display(output)

Output()

## Notes

- The vector branch calls Supabase RPC `match_gi_factoids` with strict positional exactness (`filter_position=<player position>`).
- Vector query embedding input is built from Flash 2.5 summary + HS scouting report + measurables + filtered skill ratings.
- If the RPC is unavailable, vector insights are skipped automatically and the rest of the pipeline still runs.
- The final synthesis model is configured as `gemini-3-flash` in `CONFIG`.
- You can tune output style by editing `build_final_prompt(...)`.

In [41]:
# Optional smoke test: verify vector RPC connectivity
sb_test = get_supabase_client()
test_vec = vector_insights_query(
    sb=sb_test,
    query_text="DB from California, good hands, 75 inches, 190 lbs, all american",
    position="DB",
    top_k=5,
    threshold=CONFIG['VECTOR_MATCH_THRESHOLD'],
)
print(json.dumps(test_vec, indent=2))

{
  "status": "skipped",
  "reason": "Vector RPC unavailable: _call_vector_rpc() missing 1 required positional argument: 'filter_state'",
  "insights": []
}
